# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 3.0
%worker_type G.2X
%number_of_workers 10

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [ ]:
df = spark.read.option("recursiveFileLookup", "true").text('s3://cdkstack-documentsbucket9ec9deb9-sbbf9n4wdhze/embeddingarchive/')

In [ ]:
dfP = spark.read.option("recursiveFileLookup", "true").text('s3://cdkstack-documentsbucket9ec9deb9-sbbf9n4wdhze/promptarchive/')

In [ ]:
df.count()

In [ ]:
dfP.count()

In [ ]:
from pyspark.sql.functions import split, col, regexp_replace, transform
df = df.withColumn("value", regexp_replace("value", r'(\[)', '')).withColumn("value", regexp_replace("value", r'(])', ''))
dfP = dfP.withColumn("value", regexp_replace("value", r'(\[)', '')).withColumn("value", regexp_replace("value", r'(])', ''))

In [ ]:
from pyspark.sql.functions import split, col
df = df.select(split(col("value"),",").alias("EmbedArray")).drop("value")
dfP = dfP.select(split(col("value"),",").alias("EmbedArray")).drop("value")

In [ ]:
df = df.withColumn("EmbedArray", transform(col("EmbedArray"), lambda x: x.cast("float")))
df = df.withColumn("EmbedArray", col("EmbedArray").cast("array<float>"))
dfP = dfP.withColumn("EmbedArray", transform(col("EmbedArray"), lambda x: x.cast("float")))
dfP = dfP.withColumn("EmbedArray", col("EmbedArray").cast("array<float>"))

In [ ]:
from pyspark.ml.linalg import Vectors

In [ ]:
from pyspark.ml.functions import array_to_vector

In [ ]:
df = df.select(array_to_vector('EmbedArray').alias('EmbedArray'))
dfP = dfP.select(array_to_vector('EmbedArray').alias('EmbedArray'))

In [ ]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
  inputCols=["EmbedArray"], outputCol="features"
)

dfTrain = assembler.transform(df).drop('EmbedArray')
dfTrainP = assembler.transform(dfP).drop('EmbedArray')

In [ ]:
from pyspark.ml.feature import PCA
pca = PCA(k=100, inputCol="features")

In [ ]:
pca.setOutputCol("pca_features")
pca_model = pca.fit(dfTrain)

In [ ]:
pca_model.setOutputCol("output")
dfPca = pca_model.transform(dfTrain)
dfPcaP = pca_model.transform(dfTrainP)

In [ ]:
import numpy as np

In [ ]:
from pyspark.ml.clustering import KMeans

In [ ]:
kmeans = KMeans(k=10)

In [ ]:
kmeans_model = kmeans.fit(dfPca)

In [ ]:
kmeans_model.setPredictionCol("newPrediction")

In [ ]:
dfKmeans = kmeans_model.transform(dfPca).select("features", "newPrediction")

In [ ]:
dfKmeansP = kmeans_model.transform(dfPcaP).select("features", "newPrediction")

In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator
evaluator = ClusteringEvaluator(predictionCol='newPrediction', featuresCol='features', \
                                metricName='silhouette', distanceMeasure='squaredEuclidean')

In [ ]:
score=evaluator.evaluate(dfKmeans)
score

In [ ]:
scoreP=evaluator.evaluate(dfKmeansP)
scoreP

In [ ]:
dfKmeansP.printSchema()

In [ ]:
dfKmeansP.head(1)[0]['newPrediction']

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType
l_clusters = kmeans_model.clusterCenters()
# Let's convert the list of centers to a dict, each center is a list of float
d_clusters = {int(i):[float(l_clusters[i][j]) for j in range(len(l_clusters[i]))] for i in range(len(l_clusters))}

# Let's create a dataframe containing the centers and their coordinates
df_centers = spark.sparkContext.parallelize([(k,)+(v,) for k,v in d_clusters.items()]).toDF(['prediction','center'])

dfKmeansP = dfKmeansP.withColumn('prediction',F.col('newPrediction').cast(IntegerType()))
dfKmeansP = dfKmeansP.join(df_centers,on='prediction',how='left')

In [ ]:
from pyspark.sql.types import FloatType
get_dist = F.udf(lambda features, center : float(features.squared_distance(center)),FloatType())
dfKmeansP = dfKmeansP.withColumn('dist',get_dist(F.col('features'),F.col('center')))

In [ ]:
dfKmeansP.printSchema()

In [ ]:
dfKmeansP.head(1)[0]['dist']

In [ ]:
from pyspark.sql.functions import mean as _mean, stddev as _stddev

df_stats = dfKmeansP.select(
    _mean(col('dist')).alias('mean'),
    _stddev(col('dist')).alias('std')
).collect()

mean = df_stats[0]['mean']
std = df_stats[0]['std']

In [ ]:
mean

In [ ]:
std